# Multi-Agent Orchestration Layer - Multi-Agent ReAct con LangGraph

**Sistema Operativo de Agentes Cognitivos para Inteligencia Competitiva**

Este notebook demuestra el módulo de orquestación multi-agente: recibe una pregunta, la procesa con tres agentes (Researcher, Fact Auditor, Writer) coordinados por un grafo LangGraph con guardrails y ciclo de rechazo y corrección, y entrega una salida lista para el Evaluation Layer.

Todo el código vive en `src/`; este notebook solo importa, ejecuta y explica.

## 1. Objetivo

- Mostrar el `AgentState` y los tres agentes por separado.
- Construir y visualizar el grafo LangGraph con enrutamiento condicional.
- Ejecutar `multi_agent_rag(question)` en modo fallback (sin dependencias externas).
- Mostrar un ejemplo **aprobado** y un ejemplo **rechazado / corregido**.
- Explicar los guardrails contra alucinaciones y su umbral configurable.
- Exportar el resultado al formato que consume el Evaluation Layer.
- Mostrar cómo conectar un retriever real del Data & Retrieval Layer sin tocar los agentes.
- Presentar la analogía conceptual con Aprendizaje por Refuerzo (no es entrenamiento real).

## 2. Entorno

Si corres esto en Google Colab, la siguiente celda clona el repo (opcional) e instala las dependencias. Si ya estás dentro del repo clonado (por ejemplo en VS Code), sáltala.

In [ ]:
# Descomentar en Colab:
# REPO_URL = "https://github.com/<usuario>/PLN_P2.git"
# import os
# if not os.path.isdir("PLN_P2"):
#     !git clone $REPO_URL
# %cd PLN_P2

!pip install -q -r requirements.txt

## 3. Imports

Todo el código importante vive en `src/`; este notebook no duplica lógica.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config import DEFAULT_CONFIG
from src.agents.state import AgentState
from src.agents.researcher import researcher_node
from src.agents.auditor import auditor_node, estimate_evidence_score, estimate_hallucination_risk
from src.agents.writer import writer_node
from src.graph.build_graph import build_agent_graph
from src.graph.routes import route_after_audit
from src.graph.visualize import visualize_graph
from src.pipeline.multi_agent_rag import multi_agent_rag
from src.pipeline.eval_format import convert_to_eval_format
from src.retriever.interface import register_retriever
from src.retriever.fallback_retriever import fallback_retrieve_context
from src.llm.fallback_llm import fallback_generate_llm_response
from src.utils.trace import print_trace, save_trace

## 4. Configuración global: fallback, LLM local o Gemini

El módulo de orquestación **no depende de un único proveedor de LLM**. `generate_llm_response(prompt)` es la interfaz común (`src/llm/interface.py`), resuelta según `provider`:

| provider   | Módulo               | Cuándo usarlo                                             |
|------------|----------------------|------------------------------------------------------------|
| `fallback` | `src/llm/fallback_llm.py` | Pruebas locales rápidas, sin GPU ni API keys.          |
| `local`    | `src/llm/local_llm.py`    | **Modo principal de arquitectura** (Mistral-7B / Llama-3-8B en 4 bits), tal como pide el enunciado. |
| `gemini`   | `src/llm/gemini_llm.py`   | Desarrollo rápido opcional sin GPU (requiere `GOOGLE_API_KEY`). |

El mismo patrón aplica al retriever: `USE_FALLBACK_RETRIEVER=True` usa `fallback_retrieve_context`; en `False`, se necesita `register_retriever()` con el retriever real del Data & Retrieval Layer (ver sección 9).

In [ ]:
DEFAULT_CONFIG

## 5. AgentState

`AgentState` (TypedDict) es el estado compartido entre los tres nodos del grafo. LangGraph lo pasa de nodo a nodo; cada nodo devuelve solo las claves que actualiza.

In [ ]:
initial_state: AgentState = {
    "question": "Cuando y por quien fue construida la Torre Eiffel?",
    "retrieved_context": [],
    "draft_answer": "",
    "evidence_list": [],
    "limitations": "",
    "audit_passed": False,
    "audit_feedback": "",
    "missing_info": "",
    "evidence_score": 0.0,
    "hallucination_risk": 1.0,
    "route_decision": "",
    "final_answer": "",
    "confidence_level": "Bajo",
    "warnings": [],
    "iterations": 0,
    "trace": [],
    "system_type": "multi_agent_react",
}
initial_state

## 6. Los tres agentes, paso a paso

### Researcher Agent
Llama a `retrieve_context(question, top_k)` y redacta un `draft_answer` apoyado únicamente en el contexto recuperado.

In [ ]:
state = dict(initial_state)
researcher_output = researcher_node(state, agent_config=DEFAULT_CONFIG)
state.update(researcher_output)
print("draft_answer:", state["draft_answer"])
print("evidence_list:", state["evidence_list"])

### Fact Auditor Agent

No le pregunta al mismo LLM si su respuesta está bien: calcula `evidence_score` y `hallucination_risk` con una heurística léxica independiente (`estimate_evidence_score`, `estimate_hallucination_risk` en `src/agents/auditor.py`) y decide `route_decision`.

In [ ]:
auditor_output = auditor_node(state, agent_config=DEFAULT_CONFIG)
state.update(auditor_output)
print("audit_passed:", state["audit_passed"])
print("evidence_score:", state["evidence_score"], "| hallucination_risk:", state["hallucination_risk"])
print("route_decision:", state["route_decision"])
print("audit_feedback:", state["audit_feedback"])

### Writer Agent

Produce la respuesta final con evidencia citada y nivel de confianza.

In [ ]:
writer_output = writer_node(state, agent_config=DEFAULT_CONFIG)
state.update(writer_output)
print(state["final_answer"])

## 7. Guardrails contra alucinaciones

Reglas aplicadas por el Fact Auditor (`src/agents/auditor.py`), con umbrales configurables en `config/settings.yaml`:

1. Sin contexto recuperado -> se rechaza.
2. `evidence_score < MIN_EVIDENCE_SCORE` (0.70 por defecto) -> se rechaza.
3. `hallucination_risk > MAX_HALLUCINATION_RISK` (0.30 por defecto) -> se rechaza.
4. En otro caso -> se aprueba.
5. Si se alcanza `MAX_ITERATIONS` sin aprobar -> se entrega igual, pero con advertencia explícita y confianza baja (`route_decision = "writer_with_warning"`).
6. Si no hay evidencia en absoluto, el Writer devuelve el mensaje estándar:
   *"No hay evidencia suficiente en el contexto recuperado para responder con seguridad."*

## 8. Grafo LangGraph

```
START -> researcher -> auditor -> route_after_audit
                                     |-- audit_passed=True --------------> writer -> END
                                     |-- rechazado, quedan iteraciones --> researcher (ciclo)
                                     '-- iterations >= MAX_ITERATIONS ---> writer (con advertencia) -> END
```

In [ ]:
compiled_graph = build_agent_graph()
print(visualize_graph(compiled_graph))

```mermaid
graph TD
    A[User Question] --> B[Researcher Agent]
    B --> C[Fact Auditor Agent]
    C -->|Approved| D[Writer Agent]
    C -->|Rejected and iterations available| B
    C -->|Max iterations reached| D
    D --> E[Final Answer]
```

## 9. Función principal: `multi_agent_rag(question)`

En modo fallback (por defecto, sin argumentos) no requiere GPU, API keys, ni la base vectorial real.

In [ ]:
aprobado = multi_agent_rag("Cuando y por quien fue construida la Torre Eiffel?")
print(aprobado["final_answer"])
print("\naudit_passed:", aprobado["audit_passed"], "| iterations:", aprobado["iterations"])
print("confidence_level:", aprobado["confidence_level"])

## 10. Ejemplo aprobado vs. ejemplo rechazado

El ejemplo anterior fue el caso **aprobado** (buen contexto, aprueba en la primera pasada). El siguiente es el caso **rechazado**: contexto pobre (activado con la palabra clave `baja_evidencia`), el Fact Auditor rechaza, el grafo vuelve al Researcher, y al agotar `MAX_ITERATIONS` el Writer entrega la respuesta con advertencia y confianza baja.

In [ ]:
rechazado = multi_agent_rag("baja_evidencia: para que fue disenada originalmente esta estructura?")
print(rechazado["final_answer"])
print("\naudit_passed:", rechazado["audit_passed"], "| iterations:", rechazado["iterations"])
print("warnings:", rechazado["warnings"])

## 11. Trazabilidad ReAct (Thought -> Action -> Observation)

Cada agente registra eventos breves y auditables, no cadenas de razonamiento extensas.

In [ ]:
print("--- Traza: ejemplo aprobado ---")
print_trace(aprobado)

print("\n--- Traza: ejemplo rechazado ---")
print_trace(rechazado)

save_trace(aprobado, "outputs/traces/trace_aprobado.json")
save_trace(rechazado, "outputs/traces/trace_rechazado.json")

## 12. Exportación para el Evaluation Layer

`convert_to_eval_format(result)` produce el formato plano que el Evaluation Layer necesita para RAGAS / LLM-as-a-Judge.

In [ ]:
eval_aprobado = convert_to_eval_format(aprobado)
eval_rechazado = convert_to_eval_format(rechazado)
eval_aprobado, eval_rechazado

## 13. Integración con el Data & Retrieval Layer

Ningún agente cambia para pasar de retriever fallback a retriever real: basta con registrar la función real (ver `docs/integration_with_retriever.md` y `examples/integration_example_retriever.py`).

In [ ]:
# from retrieval_pipeline import retrieve_context  # función real del Data & Retrieval Layer
# register_retriever(retrieve_context)
# resultado_real = multi_agent_rag("...", retriever=retrieve_context)

# Con el retriever fallback explícito (equivalente al comportamiento por defecto):
resultado_fallback = multi_agent_rag(
    question="Como ayudan los guardrails a reducir alucinaciones en un sistema RAG?",
    retriever=fallback_retrieve_context,
    llm=fallback_generate_llm_response,
)
print(resultado_fallback["final_answer"])

## 14. Analogía conceptual con Aprendizaje por Refuerzo

**No se entrena ningún agente con RL.** Esta es solo una analogía pedagógica para leer el grafo con el vocabulario del curso (agente, estado, acción, observación, política, recompensa, episodio):

| Concepto RL | Equivalente en este sistema |
|---|---|
| Estado | `AgentState`: pregunta, contexto, borrador, auditoría, iteraciones. |
| Acción | Recuperar contexto, redactar borrador, auditar, aprobar/rechazar, redactar respuesta final. |
| Observación | Contexto recuperado y `audit_feedback` del Fact Auditor. |
| Política de control | `route_after_audit(state)` — función determinista, no aprendida. |
| Recompensa proxy | Mejora de faithfulness / answer relevance y reducción de `hallucination_risk`, medida externamente por el Evaluation Layer. |
| Episodio | Una ejecución completa de `multi_agent_rag(question)`, de START a END. |

Ver `docs/architecture.md` sección 6 para el detalle completo.

## 15. Conclusiones

- El Multi-Agent Orchestration Layer funciona de punta a punta en modo fallback, sin dependencias externas.
- Los guardrails cuantitativos (independientes del LLM) reducen el riesgo de aceptar respuestas no sustentadas.
- El ciclo de rechazo y corrección, acotado por `MAX_ITERATIONS`, siempre entrega una respuesta (con advertencia si no se pudo aprobar).
- La interfaz `retrieve_context` / `generate_llm_response` permite conectar el Data & Retrieval Layer real y un LLM local cuantizado (o Gemini) sin modificar ningún agente.
- La salida de `convert_to_eval_format()` queda lista para el Evaluation Layer.